In [ ]:
import pandas as pd
import os
from pathlib import Path
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
import numpy as np
import seaborn as sns

In [ ]:
BASE_DIR = Path(
    '..'
)
DATA_DIR = BASE_DIR / "data"

In [ ]:
TARGET = "mass_balance_annual"
CATEGORICAL_FEATURES = ["satellite"]
NUMERICAL_FEATURES = [
    "sla_norm",
    "elev_mean",
    "slope_mean",
    "aspect_mean",
    "snow_fraction",
    "B2",
    "B3",
    "B4",
    "B8",
    "B11",
    "B12",
    "q1h_temp",
    "q2h_temp",
    "q3h_temp",
    "q4h_temp",
    "q1h_prec",
    "q2h_prec",
    "q3h_prec",
    "q4h_prec",
]

In [ ]:
path = DATA_DIR / "processed" / f"glacier_ml_dataset_v1.0.parquet"
df = pd.read_parquet(path)
df[CATEGORICAL_FEATURES] = df[CATEGORICAL_FEATURES].astype("category")

In [ ]:
df

In [ ]:
df.observation_start.min().year

In [ ]:
y = df[TARGET]
y

In [ ]:
y.mean()

In [ ]:
y.hist()

In [ ]:
df[[TARGET, 'sla']].mean().to_dict()

In [ ]:
X = df[NUMERICAL_FEATURES]

In [ ]:
dtrain = xgb.DMatrix(X, label=y)

In [ ]:
bst = xgb.train({}, dtrain)

In [ ]:
xgb.plot_importance(bst, importance_type='cover')

In [ ]:
bst.get_fscore()

In [ ]:
test = bst.get_score(importance_type='gain')

In [ ]:
test['test'] = 'what'

In [ ]:
importances = []
weight = bst.get_score(importance_type="weight")
weight["importance_type"] = "weight"

gain = bst.get_score(importance_type="gain")
gain["importance_type"] = "gain"

cover = bst.get_score(importance_type="cover")
cover["importance_type"] = "cover"

importances.append(weight)
importances.append(gain)
importances.append(cover)

df = pd.DataFrame(importances)
df

In [ ]:
df_transformed = df.set_index('importance_type').T.reset_index()
df_transformed = df_transformed.rename(columns={'index': 'feature'})
df_transformed.columns.name = None
df_transformed

In [ ]:
rfr = RandomForestRegressor()

In [ ]:
rfr.fit(X, y)

In [ ]:
rfr.feature_importances_

In [ ]:
rfr.feature_names_in_

In [ ]:
X.columns.values

In [ ]:
result = permutation_importance(
    rfr, X, y, n_repeats=10, random_state=42, n_jobs=-1
)
forest_importances = pd.DataFrame([X.columns.values, result.importances_mean]).T
forest_importances.columns = ['feature', 'importance']

In [ ]:
forest_importances
forest_importances.sort_values('importance', ascending=False)

In [ ]:
sns.barplot(forest_importances.sort_values('importance', ascending=False), x='importance', y='feature', orient='h')